# Drought Forecast Verification — Template Notebook

This notebook demonstrates the full `drought-forecast-eval` workflow:

1. **Download** (optional) fresh data from BGDI
2. **Load** forecast archive and observation archive
3. **Align** observations to the forecast grid
4. **Build** a reference forecast (climatology or persistence)
5. **Compute** verification scores
6. **Visualise** skill curves and regional maps
7. **Export** results (NetCDF, CSV, HTML report)

Copy this notebook and adapt the paths / settings for your specific experiment.

---
**Docs:** `docs/` | **Source:** `drought_forecast_eval/` | **Issues / MRs:** GitLab repo

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

# ── EDIT THESE PATHS ──────────────────────────────────────────────────────
ARCHIVE_DIR   = Path("data/zipfiles")               # flat dir of daily ZIPs
HISTORIC_ZIP  = Path("data/historic.csv.zip")      # full observation archive
SHAPEFILE     = Path('shapefiles/BAFU_Grossregionen_v3.shp')
OUTPUT_DIR    = Path("output")

DATE_RANGE    = ("2025-07-01", "2025-12-31")        # verification window
REGIONS       = "all"                                # or e.g. [1, 5, 12, 31]
VARIABLES     = "all"                                # or e.g. ["spi_1m", "cdi"]
CDI_THRESHOLD = 2                                    # CDI ≥ threshold = drought event
# ──────────────────────────────────────────────────────────────────────────

## 1. (Optional) Download fresh data from BGDI

In [ ]:
from drought_forecast_eval.io.downloader import download_archive, download_historic

# Uncomment to download:
# download_archive(ARCHIVE_DIR, start=DATE_RANGE[0], end=DATE_RANGE[1])
# download_historic(HISTORIC_ZIP)

## 2. Load data

In [ ]:
from drought_forecast_eval import load_forecasts, load_observations

region_ids = None if REGIONS == "all" else REGIONS

ds_fcst = load_forecasts(ARCHIVE_DIR, date_range=DATE_RANGE, region_ids=region_ids)
ds_obs  = load_observations(HISTORIC_ZIP, region_ids=region_ids)

In [ ]:
ds_fcst

In [ ]:
ds_obs

## 3. Align observations to the forecast grid

In [ ]:
from drought_forecast_eval import align_obs_to_forecast

ds_merged = align_obs_to_forecast(ds_fcst, ds_obs)
print("Merged dataset dims:", dict(ds_merged.dims))

In [ ]:
ds_merged

## 4. Build reference forecast

Choose `"climatology"`, `"persistence"`, or `"both

In [ ]:
from drought_forecast_eval import climatology_reference, persistence_reference

# --- Climatology (week-of-year ± 4 weeks) ---
ds_ref_clim = climatology_reference(ds_obs, ds_fcst, week_window=4)

# --- Persistence (last observed value before each FRT) ---
ds_ref_pers = persistence_reference(ds_obs, ds_fcst)

print("Climatology reference variables:", list(ds_ref_clim.data_vars)[:5], "...")

## 5 Explorative Data Analysis

In [ ]:
import matplotlib.pylab as plt

%matplotlib inline
plt.show()

In [ ]:
RID = 31 # Region Identifier

In [ ]:
from drought_forecast_eval.visualization.skill_curves import plot_forecast_plume

In [ ]:
fig, ax = plot_forecast_plume(ds_merged=ds_merged,
                    ds_ref_clim=ds_ref_clim,
                    ds_ref_pers=ds_ref_pers,
                    rid=RID,
                    var="spi_1m")

In [ ]:
fig, ax = plot_forecast_plume(ds_merged=ds_merged,
                    ds_ref_clim=ds_ref_clim,
                    ds_ref_pers=ds_ref_pers,
                    rid=RID,
                    var="precip_sum_1m")

In [ ]:
fig, ax = plot_forecast_plume(ds_merged=ds_merged,
                    ds_ref_clim=ds_ref_clim,
                    ds_ref_pers=ds_ref_pers,
                    rid=RID,
                    var="soil_moisture_ufc")

## 6. Compute scores

In [ ]:
from drought_forecast_eval import compute_scores

ds_scores = compute_scores(
    ds_merged,
    variables=VARIABLES,
    scores="default",
    ds_ref=ds_ref_clim,
    cdi_threshold=CDI_THRESHOLD,
)

print("Score variables:", list(ds_scores.data_vars))
ds_scores

In [ ]:
ds_scores

### 5a. Add custom metrics
You can register your own scoring function and include it by name:

In [ ]:
import xarray as xr
from drought_forecast_eval import register_score

def my_custom_score(fcst: xr.DataArray, obs: xr.DataArray, **kwargs) -> xr.DataArray:
    """Example: mean signed error."""
    return (fcst - obs).mean(dim=kwargs.get("reduce_dims", []))

register_score("signed_error", my_custom_score)

ds_scores_custom = compute_scores(
    ds_merged,
    variables=["spi_1m", "cdi"],
    scores=["mae", "signed_error"],
)
ds_scores_custom

## 6. Visualise results

In [ ]:
import matplotlib.pyplot as plt
from drought_forecast_eval.visualization import plot_skill_vs_lead

fig = plot_skill_vs_lead(
    ds_scores,
    variables=["spi_1m", "spi_3m", "spi_24m", "hydro_index"],
    score="mae",
    title="MAE by lead week",
)
plt.show()

In [ ]:
ds_scores

In [ ]:
help(plot_skill_vs_lead)

In [ ]:
import matplotlib.pyplot as plt
from drought_forecast_eval.visualization import plot_skill_vs_lead



for _score in ["correlation","bias","mae"]:
    fig = plot_skill_vs_lead(
        ds_scores, #.sel(drought_region_id=31),
        variables=["precip_1m_index", "precip_3m_index", "precip_24m_index", "hydro_index", "soil_moisture_index", "cdi"],
        aggregate_regions=True,
        score=f"{_score}",
        title=f"{_score} by lead week",
    )
    plt.show()

In [ ]:
# RPSS (requires reference forecast)
ds_scores_w_skill = compute_scores(
    ds_merged,
    variables=["spi_1m", "spi_3m", "spi_24m", "soil_moisture_ufc", "hydro_index"],
    scores=["mae_skill"],
    ds_ref=ds_ref_clim,
)

fig2 = plot_skill_vs_lead(
    ds_scores_w_skill,
    variables=["spi_1m", "spi_3m","soil_moisture_ufc","spi_24m"],
    score="mae_skill",
    zero_line=True,
    title="MAE SKILL vs climatology reference",
)
plt.show()

In [ ]:
ds_merged

In [ ]:
# Regional map (requires shapefile; falls back to bar chart if not provided)
from drought_forecast_eval.visualization import plot_region_map

fig3 = plot_region_map(
    ds_scores_w_skill,
    score_variable="spi_1m_mae_skill",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="plasma",
    vmin=0,
    vmax=1
)
fig3 = plot_region_map(
    ds_scores_w_skill,
    score_variable="spi_3m_mae_skill",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="plasma",
    vmin=0,
    vmax=1
)
fig3 = plot_region_map(
    ds_scores_w_skill,
    score_variable="soil_moisture_ufc_mae_skill",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="plasma",
    vmin=0,
    vmax=1
)


plt.show()

In [ ]:
# Experiment comparison: period A vs period B
from drought_forecast_eval.visualization import plot_experiment_diff

# Example: compare first half vs second half of the year
DATE_RANGE_A = ("2025-07-01", "2025-08-29")
DATE_RANGE_B = ("2025-09-01", "2025-10-31")

ds_fcst_a = load_forecasts(ARCHIVE_DIR, date_range=DATE_RANGE_A, region_ids=region_ids)
ds_fcst_b = load_forecasts(ARCHIVE_DIR, date_range=DATE_RANGE_B, region_ids=region_ids)

ds_merged_a = align_obs_to_forecast(ds_fcst_a, ds_obs)
ds_merged_b = align_obs_to_forecast(ds_fcst_b, ds_obs)

ds_ref_a = climatology_reference(ds_obs, ds_fcst_a)
ds_ref_b = climatology_reference(ds_obs, ds_fcst_b)

ds_scores_a = compute_scores(ds_merged_a, variables=["spi_1m", "cdi"], scores=["mae"], ds_ref=ds_ref_a)
ds_scores_b = compute_scores(ds_merged_b, variables=["spi_1m", "cdi"], scores=["mae"], ds_ref=ds_ref_b)

fig4 = plot_experiment_diff(
    ds_scores_a, ds_scores_b,
    variables=["spi_1m", "cdi"],
    score="mae",
    label_a="H1 2025",
    label_b="H2 2025",
)
plt.show()

## 7. Export results

In [ ]:
from drought_forecast_eval import ReportBuilder

builder = ReportBuilder(
    archive_dir=ARCHIVE_DIR,
    historic_path=HISTORIC_ZIP,
    date_range=DATE_RANGE,
    regions=REGIONS,
    variables=VARIABLES,
    scores="default",
    reference="climatology",
    cdi_threshold=CDI_THRESHOLD,
    shapefile_path=SHAPEFILE,
    output_dir=OUTPUT_DIR,
)

# Run the full pipeline and write all outputs
outputs = builder.run()
print("Outputs:")
for key, path in outputs.items():
    print(f"  {key}: {path}")

In [ ]:
# Or export individually:
# builder.export_netcdf(OUTPUT_DIR / "scores.nc")
# builder.export_csv(OUTPUT_DIR / "scores_summary.csv")
# builder.export_html(OUTPUT_DIR / "report.html")